In [ ]:
# --- PARAMETERS ---
BATCH_SIZE = 64  # Increased slightly for stability
NUM_ITERATIONS = 20000  # Set to 20000+ for full run, 1000 for quick test
USE_BALANCED_TRAINING = True

# Environment
WIDTH = 9
HEIGHT = 9
NUM_AGENTS = 4

# Model Architecture
LEARNING_RATE = 0.0001
CONV_CHANNELS = [32, 64, 64]
KERNEL_SIZE = 3
POOL = False                # <--- Control Pooling here
MLP_HIDDEN_FEATURES = 256
NUM_MLP_HIDDEN_LAYERS = 2

# File Paths
# Input path for states (adjust relative path as needed for your cluster)
STATES_FILE_PATH = f"centralized_model{WIDTH}x{HEIGHT}_agents{NUM_AGENTS}/trained_states_centralized.pkl"
# Output path for the trained model
MODEL_SAVE_PATH = f"reward_model_{WIDTH}x{HEIGHT}_{NUM_AGENTS}a_pool{POOL}"

DEBUG = False # Set True to limit data for extremely fast debugging

In [ ]:
import sys
import os
import pickle
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path

# Add path to orchard root (adjust if necessary)
sys.path.append('../../../')

from tadd_helpers.env_functions import State
from models.cnn import CNNDecentralized

# Import the new class you modified in cnn.py

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Ensure output directory exists
Path(MODEL_SAVE_PATH).parent.mkdir(parents=True, exist_ok=True)

In [ ]:
MODEL_SAVE_PATH = Path(MODEL_SAVE_PATH)
if not MODEL_SAVE_PATH.exists():
    MODEL_SAVE_PATH.mkdir(parents=True)


In [ ]:
def get_decentralized_reward_when_picked(picked: bool, picker_index, picker_pos, all_agents) -> dict[int, float]:
    """Return decentralized reward of all agents

    Args:
        picked (bool): Whether an agent has picked an apple
        picker_index: _description_
        picker_pos: _description_
        all_agents: dict[int, tuple]: Mapping from agent index to (x, y) position
    """
    res: dict[int, float] = {}
    
    if not picked:
        for agent_idx in all_agents.keys():
            res[agent_idx] = 0.0
        return res

    res[picker_index] = -1

    # Calculate distances from ALL agents to the picker
    all_agent_positions = np.array(list(all_agents.values()))
    distances = np.linalg.norm(all_agent_positions - np.array(picker_pos), axis=1)
    sum_of_distances = np.sum(distances)
    
    for agent_idx in all_agents.keys():
        if agent_idx == picker_index:
            continue
        agent_pos = all_agents[agent_idx]
        agent_distance = np.linalg.norm(np.array(agent_pos) - np.array(picker_pos))
        if sum_of_distances == 0:
            res[agent_idx] = 0.0
        else:
            res[agent_idx] = 2 * agent_distance / sum_of_distances
    return res

def get_picker_index_and_pos_from_state(state: State):
    """Extract the picker index and position from the state. 

    Args:
        state (State): The current environment state
    Returns:
        picked: bool: Whether an agent has picked an apple
        picker_index (int): The index of the picker agent
        picker_pos (tuple): The (x, y) position of the picker agent 
    """
    picked = False
    picker_index = -1
    picker_pos = (-1, -1)
    for agent_idx, agent_pos in state._agents.items():
        # check if the agent_pos is in the apples nd array
        if state._apples[agent_pos] >= 1:
            picked = True
            picker_index = agent_idx
            picker_pos = agent_pos
            break
    return picked, picker_index, picker_pos

def get_decentralized_reward_from_state(state: State) -> dict[int, float]:
    """Calculate the decentralized reward for the self-agent based on the state.

    Args:
        state (State): The current environment state
    Returns:
        dict[int, float]: The decentralized rewards for all agents
    """
    picked, picker_index, picker_pos = get_picker_index_and_pos_from_state(state)
    all_agents = state._agents
    rewards = get_decentralized_reward_when_picked(picked, picker_index, picker_pos, all_agents)
    return rewards

In [ ]:
# --- Data Loading & Processing ---
if not os.path.exists(STATES_FILE_PATH):
    raise FileNotFoundError(f"Could not find states file at: {STATES_FILE_PATH}")

with open(STATES_FILE_PATH, "rb") as f:
    states = pickle.load(f)

print(f"Loaded {len(states)} states.")

# Buckets for Training Strategy
data_zero_reward = []
data_interesting = [] # >0 or -1
data_all_flat = []    # For "Natural" training

print("Processing states into training buckets...")
states_to_process = states[:2000] if DEBUG else states

for state in tqdm(states_to_process):
    rewards = get_decentralized_reward_from_state(state)
    raw_state = {"agents": state.agents, "apples": state.apples}
    
    for agent_id, r in rewards.items():
        item = (raw_state, state.agent_position(agent_id), r)
        
        data_all_flat.append(item) # Keep everything for natural sampling
        
        if r == 0:
            data_zero_reward.append(item)
        else:
            data_interesting.append(item)

print(f"Total Samples: {len(data_all_flat)}")
print(f"  - Zero Reward: {len(data_zero_reward)}")
print(f"  - Interesting: {len(data_interesting)}")

In [ ]:
# --- CELL 5: FIXED MODEL DEFINITION ---
import torch.nn as nn

class CoordCNNDecentralized(CNNDecentralized):
    def __init__(
        self,
        height,
        width,
        alpha,
        mlp_hidden_features=256,
        num_mlp_hidden_layers=2,
        conv_channels=[16, 32],
        kernel_size=3,
        pool=False,
    ):
        # 1. Initialize Parent (This builds the WRONG MLP, but sets up other stuff)
        super().__init__(
            height=height,
            width=width,
            alpha=alpha,
            mlp_hidden_features=mlp_hidden_features,
            num_mlp_hidden_layers=num_mlp_hidden_layers,
            conv_channels=conv_channels,
            kernel_size=kernel_size,
        )

        # 2. Rebuild Conv Layers (To support 5 channels and optional pooling)
        self.conv_layers = torch.nn.ModuleList()
        in_channels = 5  # CoordConv = 5 Channels

        current_height, current_width = height, width

        for out_channels in conv_channels:
            padding = (kernel_size - 1) // 2
            self.conv_layers.append(
                torch.nn.Conv2d(
                    in_channels, out_channels, kernel_size=kernel_size, padding=padding
                )
            )
            self.conv_layers.append(torch.nn.ReLU())
            
            # Logic: Apply pooling ONLY if requested AND dimensions allow
            if pool and current_width >= 2 and current_height >= 2:
                self.conv_layers.append(torch.nn.MaxPool2d(kernel_size=2, stride=2))
                current_height //= 2
                current_width //= 2
            in_channels = out_channels

        # --- FIX START: REBUILD MLP HEAD ---
        # 3. Recalculate the Flat Size
        # We run a dummy pass with the NEW conv layers to get the correct size (e.g., 5184)
        conv_output_size = self._get_conv_output_size(5, height, width)
        
        # 4. Rebuild the MLP Sequence
        layers = []
        input_features = conv_output_size
        for _ in range(num_mlp_hidden_layers):
            layers.append(nn.Linear(input_features, mlp_hidden_features))
            layers.append(nn.ReLU())
            input_features = mlp_hidden_features
        layers.append(nn.Linear(input_features, 1))
        
        self.mlp_head = nn.Sequential(*layers)
        # --- FIX END ---

        # 5. Re-init optimizer because parameters changed
        self.optimizer = torch.optim.AdamW(self.parameters(), lr=alpha, amsgrad=True)

        # 6. Create the Meshgrid
        x_coords = np.linspace(-1, 1, width)
        y_coords = np.linspace(-1, 1, height)
        self.xv, self.yv = np.meshgrid(x_coords, y_coords)
        self.xv = self.xv.astype(np.float32)
        self.yv = self.yv.astype(np.float32)
        
        # 7. Ensure everything moves to the correct device
        self.to(device)

    def raw_state_to_nn_input(self, state: dict, **kwargs) -> np.ndarray:
        agent_pos = kwargs.get("agent_pos")
        if agent_pos is None:
            raise ValueError("CoordCNNDecentralized requires 'agent_pos' in kwargs")
            
        agents_map = state["agents"]
        apples_map = state["apples"]

        channel_apples = apples_map.astype(np.float32)

        self_agent_map = np.zeros_like(agents_map, dtype=np.float32)
        self_agent_map[agent_pos[0], agent_pos[1]] = 1.0

        channel_others = agents_map.astype(np.float32) - self_agent_map

        cnn_state = np.stack(
            [channel_apples, channel_others, self_agent_map, self.xv, self.yv], axis=0
        )

        return cnn_state.astype(np.float32)

# Init Model
model = CoordCNNDecentralized(
    height=HEIGHT,
    width=WIDTH,
    alpha=LEARNING_RATE,
    mlp_hidden_features=MLP_HIDDEN_FEATURES,
    num_mlp_hidden_layers=NUM_MLP_HIDDEN_LAYERS,
    conv_channels=CONV_CHANNELS,
    kernel_size=KERNEL_SIZE,
    pool=POOL 
)

print(f"Model Initialized on {device}")
# Debug Print to verify shapes
dummy_in = torch.zeros(1, 5, HEIGHT, WIDTH).to(device)
dummy_out = model(dummy_in)
print(f"Verification Forward Pass Shape: {dummy_out.shape}")

In [ ]:
# --- Training Loop ---
losses = []
print(f"Starting Training for {NUM_ITERATIONS} iterations...")

for i in tqdm(range(NUM_ITERATIONS)):
    model.optimizer.zero_grad()
    
    batch_states = []
    batch_rewards = []
    
    if USE_BALANCED_TRAINING:
        # Strategy A: Force 50/50 Split
        half_batch = BATCH_SIZE // 2
        for _ in range(half_batch):
            s, pos, r = random.choice(data_zero_reward)
            batch_states.append(model.raw_state_to_nn_input(s, agent_pos=pos))
            batch_rewards.append(r)
        for _ in range(half_batch):
            s, pos, r = random.choice(data_interesting)
            batch_states.append(model.raw_state_to_nn_input(s, agent_pos=pos))
            batch_rewards.append(r)
    else:
        # Strategy B: Natural Sampling (92% Zeros)
        batch_items = random.sample(data_all_flat, BATCH_SIZE)
        for s, pos, r in batch_items:
            batch_states.append(model.raw_state_to_nn_input(s, agent_pos=pos))
            batch_rewards.append(r)

    # Train Step
    states_tensor = torch.tensor(np.stack(batch_states), dtype=torch.float32).to(device)
    rewards_tensor = torch.tensor(np.array(batch_rewards), dtype=torch.float32).to(device)
    
    preds = model(states_tensor).squeeze(1)
    loss = torch.nn.functional.mse_loss(preds, rewards_tensor)
    
    loss.backward()
    model.optimizer.step()
    
    losses.append(loss.item())


In [ ]:

torch.save(model.state_dict(), MODEL_SAVE_PATH / "model.pt")
plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.title("Training Loss")
plt.show()

In [ ]:
# --- Comprehensive Evaluation (All States) ---
model.eval()

# We categorize results during evaluation to get granular metrics
results = {
    "zero": {"true": [], "pred": []},
    "picker": {"true": [], "pred": []}, # -1
    "distance": {"true": [], "pred": []} # > 0
}

print("Evaluating on FULL Dataset (Natural Distribution)...")

# Use data_all_flat to simulate the real world distribution
eval_batch_size = 1000
eval_source = data_all_flat 

with torch.no_grad():
    for i in tqdm(range(0, len(eval_source), eval_batch_size)):
        batch = eval_source[i : i + eval_batch_size]
        
        batch_states = []
        batch_r = []
        
        for s, pos, r in batch:
            batch_states.append(model.raw_state_to_nn_input(s, agent_pos=pos))
            batch_r.append(r)
            
        inp = torch.tensor(np.stack(batch_states), dtype=torch.float32).to(device)
        out = model(inp).squeeze(1).cpu().numpy()
        
        # Sort into buckets for metrics
        for t, p in zip(batch_r, out):
            if t == 0:
                results["zero"]["true"].append(t)
                results["zero"]["pred"].append(p)
            elif t == -1:
                results["picker"]["true"].append(t)
                results["picker"]["pred"].append(p)
            else:
                results["distance"]["true"].append(t)
                results["distance"]["pred"].append(p)

# --- Compute Metrics ---
print("\n=== FINAL METRICS ===")

# 1. Zero Reward (Hallucination Check)
z_true = np.array(results["zero"]["true"])
z_pred = np.array(results["zero"]["pred"])
z_mae = np.mean(np.abs(z_true - z_pred))
print(f"[Zero Reward] Count: {len(z_true)}")
print(f"[Zero Reward] MAE: {z_mae:.5f} (Lower is better)")

# 2. Picker Reward (-1)
p_true = np.array(results["picker"]["true"])
p_pred = np.array(results["picker"]["pred"])
p_mae = np.mean(np.abs(p_true - p_pred))
print(f"\n[Picker Reward -1] Count: {len(p_true)}")
print(f"[Picker Reward -1] MAE: {p_mae:.5f}")

# 3. Distance Reward (>0) - The Hard Part
d_true = np.array(results["distance"]["true"])
d_pred = np.array(results["distance"]["pred"])
d_mae = np.mean(np.abs(d_true - d_pred))
d_mape = np.mean(np.abs(d_true - d_pred) / d_true) * 100
d_corr = np.corrcoef(d_true, d_pred)[0, 1]

print(f"\n[Distance Reward >0] Count: {len(d_true)}")
print(f"[Distance Reward >0] MAE: {d_mae:.5f}")
print(f"[Distance Reward >0] MAPE: {d_mape:.2f}%")
print(f"[Distance Reward >0] Correlation: {d_corr:.4f}")

# --- Plotting Distance Regression ---
plt.figure(figsize=(8, 8))
plt.scatter(d_true, d_pred, alpha=0.2, s=10)
plt.plot([0, max(d_true)], [0, max(d_true)], 'r--', label="Perfect")
plt.xlabel("True Distance Reward")
plt.ylabel("Predicted Reward")
plt.title(f"Distance Prediction (MAPE={d_mape:.1f}%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()